# 032 — GM record selection, AvgSA([0, 3]) — Stage 1 (compute)

The slow selection step. Runs the configurable multi-round selection + optimisation engine
(`build_final_ensembles`) to build the final per-`(site, iml)` record ensembles conditioned on
**AvgSA([0, 3])**, using the shared `SELECTION_CONFIG` (4 rounds dropping progressively more
causal-parameter bounds; round 4 is a shuffled-DB best-of retry).

**Incremental** — like nb `031`, this notebook calls `find_stale_stripes` and only (re)selects the
stripes that are missing or stale on disk (the batch nb `031` just built gcim for). The round loop is
per-`(site, iml)` independent, so selecting a subset is equivalent to selecting those keys within the
full set. The batch is then **merged into `AvgSA_03_final_ensembles.pickle`** so that file stays the
COMPLETE `(site, iml)` set (it is read as complete by `summarise_record_availability.py` and
provenance-stamped by nb `040`). The canonical per-stripe record is written by nb `033`; the
per-round stage caches here are per-batch scratch. Set `FORCE_RECOMPUTE = True` to reselect every
wanted stripe.

**Upstream**:

| Input | Source |
|---|---|
| GCIM target distributions (complete set) | `gcim_dist_AvgSA_03.pickle` (nb `031`) |
| Combined selection GM database | `cfg["proc_data"]["gm_database"]` |
| IML-based disaggregations + poe stats | `cfg["proc_data"]["AvgSA_03_disagg_data_gm_selection"]`, `cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"]` |
| Site model | `cfg["hazard_models"]["eshm20_wp1_site_model"]` |
| GMM logic tree + correlation flatfiles | loaded inside `setup_AvgSA03_gcim_gm_selection()` |

**Downstream** — `033-gm_selection_AvgSA_03_stage2_postprocess.ipynb` reads the complete
`AvgSA_03_final_ensembles.pickle`, writes the per-stripe pickles + manifests for the stale ones, and
re-verifies provenance.

**Run order** — run top to bottom (after `031`), then `033`. First run over a fresh results folder
selects everything (~an hour); subsequent "added a few IMLs" runs select only the new stripes.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
import numpy as np
import pandas as pd

from phd_project.config.config import load_config
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    build_final_ensembles,
    stripe_input_fingerprint,
    find_stale_stripes,
)
from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA03_gm_selection import (
    setup_AvgSA03_gcim_gm_selection,
    SELECTION_CONFIG,
    stripe_source_fps,
)


cfg = load_config()

In [ ]:
# canonical Stage-1 output (merged each run to stay the COMPLETE (site, iml) set)
final_ensembles_fp = cfg["proc_data"]["gm_selection"] / "AvgSA_03_final_ensembles.pickle"

# intermediate stage caches, one per round (provenance-guarded; per-batch scratch).
# Named {im}_rd{ii}_{selection|optimisation}.pickle.
GMS = cfg["proc_data"]["gm_selection"]
IM = "AvgSA_03"
n_rounds = len(SELECTION_CONFIG["round_unbounded"])
stage_fps = {
    "select":   [GMS / f"{IM}_rd{ii}_selection.pickle"    for ii in range(1, n_rounds + 1)],
    "optimise": [GMS / f"{IM}_rd{ii}_optimisation.pickle" for ii in range(1, n_rounds + 1)],
}

# inputs used for provenance fingerprinting (hashed by their file bytes)
gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_03.pickle"
source_fps = {
    "gm_db_file":        cfg["proc_data"]["gm_database"],
    "gcim_file":         gcim_dist_fp,
    "disagg_data_file":  cfg["proc_data"]["AvgSA_03_disagg_data_gm_selection"],
    "disagg_stats_file": cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"],
    "site_model_file":   cfg["hazard_models"]["eshm20_wp1_site_model"],
}

# The selection scheme (rounds, bounds, shuffle, seeds, rng_seed) now lives in the
# shared SELECTION_CONFIG (setup_AvgSA03_gm_selection.py) so 031/032/033 fingerprint
# each stripe identically. See that constant for the round-by-round description.

# other stuff:
rng_seed = SELECTION_CONFIG["rng_seed"]

# Escape hatch: force-reselect ALL wanted stripes, ignoring the per-stripe manifests
# (and bypassing every stage's staleness guard). Leave False for normal incremental
# runs: only the stale/new (site, iml) are (re)selected.
FORCE_RECOMPUTE = False

In [ ]:
# set up the record selection
site_iml_disaggs, disagg_stats, site_model, basic_selection_ctx, gm_db = setup_AvgSA03_gcim_gm_selection()

# Incremental cache: (re)select only stripes that are missing or stale on disk. Uses
# the same per-stripe fingerprint as 031/033 (excludes the IML subset JSON), so this
# matches the batch nb 031 just built gcim for.
RESULT_FOLDER = cfg["results"]["AvgSA_03_record_selection"]
source_fps_stripe = stripe_source_fps()
wanted = list(site_iml_disaggs.keys())
fp_fn = lambda s, i: stripe_input_fingerprint(
    s, i, source_fps_stripe, basic_selection_ctx, SELECTION_CONFIG)
to_compute, valid = find_stale_stripes(wanted, RESULT_FOLDER, fp_fn)
print(f"{len(wanted)} wanted (site, iml): {len(valid)} already valid, "
      f"{len(to_compute)} to (re)select.")

# load the (complete) gcim distributions written by nb 031
if gcim_dist_fp.is_file():
    with open(gcim_dist_fp, "rb") as file:
        gcim_dists = pickle.load(file)
    print(f"Loaded gcim for {len(gcim_dists)} (site, iml).")
else:
    gcim_dists = {}
    print("No GCIM distribution data found — run nb 031 first.")

## Stage 1 - Build final ensembles (slow compute)

Runs the configurable multi-round selection + optimisation engine and saves the canonical
`AvgSA_03_final_ensembles.pickle` (+ a `.manifest.json` provenance sidecar). Each round
drops progressively more causal-parameter bounds (`round_unbounded`), and each step is
provenance-cached: if an input (gm_db, gcim distributions, disagg, site model, params,
rng_seed, pickagm version, round config) changed since the cache was written, the stage is
recomputed instead of silently reusing stale data - set `FORCE_RECOMPUTE = True` to rebuild.

Rounds 1–3 (`force_optimisation = [False, True, False]`) reproduce the legacy AvgSA_03 result
exactly. **Round 4** re-optimises any `(site, poe)` still failing after round 3 over
`n_shuffles = 5` shuffled database orderings and keeps the best-scoring ensemble per
`(site, poe)` (replaced only if it passes or scores strictly better). Each round prints a
`[bounded: … | unbounded: …]` bounds label plus a selection-phase and optimisation-phase
breakdown; cached stages print `[cache] ... loaded`.

Post-processing (result pickles, plots, download/convert CSVs) lives in the separate,
fast notebook **`033-gm_selection_AvgSA_03_stage2_postprocess.ipynb`**.

In [ ]:
# Force override: reselect every wanted stripe (ignores the per-stripe manifests).
if FORCE_RECOMPUTE:
    to_compute = list(wanted)

# Capture the prior COMPLETE final_ensembles before build_final_ensembles overwrites
# the file with just this batch. On a forced full rebuild we start clean.
prior_final = {}
if final_ensembles_fp.is_file() and not FORCE_RECOMPUTE:
    with open(final_ensembles_fp, "rb") as f:
        prior_final = pickle.load(f)

# Subset to the stale batch and run the multi-round engine over just those keys. The
# round loop is per-(site, iml) independent, so selecting a subset is equivalent to
# selecting them within the full set. gcim for every batch key must be present (built
# by nb 031); a missing key means 031 was not run for this batch.
batch = {k: site_iml_disaggs[k] for k in to_compute}
missing = [k for k in batch if k not in gcim_dists]
if missing:
    raise KeyError(
        f"{len(missing)} stale (site, iml) have no gcim — run nb 031 first. "
        f"e.g. {missing[:5]}")

if batch:
    batch_final = build_final_ensembles(
        batch,
        disagg_stats,
        gcim_dists,
        gm_db,
        basic_selection_ctx,
        site_model,
        source_fps=source_fps,
        stage_fps=stage_fps,
        output_fp=final_ensembles_fp,
        round_unbounded=SELECTION_CONFIG["round_unbounded"],
        force_optimisation=SELECTION_CONFIG["force_optimisation"],
        shuffle=SELECTION_CONFIG["shuffle"],
        n_shuffles=SELECTION_CONFIG["n_shuffles"],
        shuffle_rng_seeds=SELECTION_CONFIG["shuffle_rng_seeds"],
        rng_seed=rng_seed,
        force_recompute=FORCE_RECOMPUTE,
    )
else:
    batch_final = {}
    print("All stripes valid — nothing to (re)select.")

# MERGE the batch into the prior complete set so final_ensembles.pickle stays the
# COMPLETE (site, iml) selection: it is consumed as complete by
# summarise_record_availability.py and provenance-stamped by nb 040. Prune any key no
# longer wanted (an IML removed from a union list). The manifest build_final_ensembles
# wrote is source-file based, so it stays valid for the merged file.
wanted_set = set(wanted)
final_ensembles = {k: v for k, v in {**prior_final, **batch_final}.items() if k in wanted_set}
if batch_final or prior_final:
    with open(final_ensembles_fp, "wb") as f:
        pickle.dump(final_ensembles, f)
print(f"final_ensembles.pickle now holds {len(final_ensembles)} (site, iml) "
      f"({len(batch_final)} (re)selected this run).")

In [ ]:
# Isolate the (site, iml) that still fail the KS test after all rounds (across the full
# merged set), mapping each to its list of failing IMs. Not pickled - just a quick look.
failing_ensembles = {
    k: v["ks_failed_ims"]
    for k, v in final_ensembles.items()
    if v is not None and not v["ks_passed"]
}

print(f"{len(failing_ensembles)} (site, iml) still failing:")
for k in failing_ensembles:
    print("  ", k, "->", failing_ensembles[k])